In [1]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils import resample

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

TensorFlow: 2.21.0
NumPy: 1.26.4


In [2]:
CSV_PATH = r"D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\dataset\raw\train.csv"

IMAGE_DIR = r"D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\dataset\raw\train_images"

MODEL_PATH = r"D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\best_resnet50.keras"

print("CSV exists:", os.path.exists(CSV_PATH))
print("Image directory exists:", os.path.exists(IMAGE_DIR))
print("Model exists:", os.path.exists(MODEL_PATH))

CSV exists: True
Image directory exists: True
Model exists: True


In [3]:
df = pd.read_csv(CSV_PATH)

df["image_path"] = df["id_code"].apply(
    lambda x: os.path.join(IMAGE_DIR, f"{x}.png")
)

df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

print("Total valid images:", len(df))

Total valid images: 3662


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["diagnosis"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["diagnosis"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 2563
Validation: 549
Test: 550


In [5]:
class_names = [
    "No DR",
    "Mild DR",
    "Moderate DR",
    "Severe DR",
    "Proliferative DR"
]

class_counts = train_df["diagnosis"].value_counts()

print("Original training distribution:")
print(class_counts.sort_index())

Original training distribution:
diagnosis
0    1263
1     259
2     699
3     135
4     207
Name: count, dtype: int64


In [6]:
target_count = class_counts.max()

print("Target samples per class:", target_count)

Target samples per class: 1263


In [7]:
balanced_parts = []

for class_id in sorted(train_df["diagnosis"].unique()):

    class_data = train_df[
        train_df["diagnosis"] == class_id
    ]

    class_oversampled = resample(
        class_data,
        replace=True,
        n_samples=target_count,
        random_state=42 + class_id
    )

    balanced_parts.append(class_oversampled)

oversampled_train_df = pd.concat(
    balanced_parts,
    ignore_index=True
)

oversampled_train_df = oversampled_train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [8]:
print("Oversampled training distribution:")
print(
    oversampled_train_df["diagnosis"]
    .value_counts()
    .sort_index()
)

Oversampled training distribution:
diagnosis
0    1263
1    1263
2    1263
3    1263
4    1263
Name: count, dtype: int64


In [9]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 8
NUM_CLASSES = 5


def load_image_tf(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)

    return image, label

In [10]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(
        mode="horizontal"
    ),
    tf.keras.layers.RandomRotation(
        factor=0.05
    ),
    tf.keras.layers.RandomZoom(
        height_factor=0.10,
        width_factor=0.10
    )
], name="augmentation")

In [11]:
def create_tf_dataset(
    dataframe,
    batch_size=8,
    shuffle=False,
    training=False
):
    image_paths = dataframe["image_path"].values
    labels = dataframe["diagnosis"].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image_tf,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:
        dataset = dataset.map(
            lambda image, label: (
                augmentation(image, training=True),
                label
            ),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.batch(batch_size)

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset

In [12]:
train_dataset = create_tf_dataset(
    oversampled_train_df,
    batch_size=BATCH_SIZE,
    shuffle=True,
    training=True
)

val_dataset = create_tf_dataset(
    val_df,
    batch_size=BATCH_SIZE,
    shuffle=False,
    training=False
)

test_dataset = create_tf_dataset(
    test_df,
    batch_size=BATCH_SIZE,
    shuffle=False,
    training=False
)

In [13]:
for images, labels in train_dataset.take(1):
    print("Training batch image shape:", images.shape)
    print("Training batch label shape:", labels.shape)

Training batch image shape: (8, 224, 224, 3)
Training batch label shape: (8,)


In [14]:
for images, labels in val_dataset.take(1):
    print("Validation batch image shape:", images.shape)
    print("Validation batch label shape:", labels.shape)

Validation batch image shape: (8, 224, 224, 3)
Validation batch label shape: (8,)


In [15]:
MODEL_DIR = r"D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\models"

if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

from resnet50_model import build_resnet50_model

In [16]:
model = build_resnet50_model(
    trainable=False
)

model.summary()

Model: "ResNet50_DR_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ retinal_image (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dr_severity (Dense)             │ (None, 5)              │        10,245 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [17]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [18]:
CHECKPOINT_PATH = r"D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\resnet50_oversampled.keras"

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    CHECKPOINT_PATH,
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

In [19]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    callbacks=[checkpoint]
)

Epoch 1/5


d:\AI-Powered-Diabetic-Retinopathy-System\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


790/790 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step - accuracy: 0.3726 - loss: 1.4952
Epoch 1: val_loss improved from None to 0.92309, saving model to D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\resnet50_oversampled.keras

Epoch 1: finished saving model to D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\resnet50_oversampled.keras
790/790 ━━━━━━━━━━━━━━━━━━━━ 332s 412ms/step - accuracy: 0.3726 - loss: 1.4952 - val_accuracy: 0.6375 - val_loss: 0.9231
Epoch 2/5
790/790 ━━━━━━━━━━━━━━━━━━━━ 0s 392ms/step - accuracy: 0.5115 - loss: 1.1654
Epoch 2: val_loss improved from 0.92309 to 0.88485, saving model to D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\resnet50_oversampled.keras

Epoch 2: finished saving model to D:\AI-Powered-Diabetic-Retinopathy-System\ml-model\saved_models\resnet50_oversampled.keras
790/790 ━━━━━━━━━━━━━━━━━━━━ 339s 429ms/step - accuracy: 0.5115 - loss: 1.1654 - val_accuracy: 0.6612 - val_loss: 0.8849
Epoch 3/5
790/790 ━━━━━━━━━

In [20]:
best_model = tf.keras.models.load_model(
    CHECKPOINT_PATH
)

print("Best oversampled model loaded successfully.")

Best oversampled model loaded successfully.


In [21]:
test_loss, test_accuracy = best_model.evaluate(
    test_dataset,
    verbose=1
)

print("\nOversampled ResNet50 Test Results")
print("----------------------------------")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

69/69 ━━━━━━━━━━━━━━━━━━━━ 27s 340ms/step - accuracy: 0.7000 - loss: 0.7267

Oversampled ResNet50 Test Results
----------------------------------
Test Loss: 0.7267
Test Accuracy: 0.7000
Test Accuracy: 70.00%


In [22]:
y_true = []
y_pred = []

for images, labels in test_dataset:
    predictions = best_model.predict(
        images,
        verbose=0
    )

    predicted_classes = np.argmax(
        predictions,
        axis=1
    )

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Number of test samples:", len(y_true))

Number of test samples: 550


In [23]:
from sklearn.metrics import classification_report

print("\nOversampled ResNet50 Classification Report")
print("---------------------------------------------")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)



Oversampled ResNet50 Classification Report
---------------------------------------------
                  precision    recall  f1-score   support

           No DR     0.9357    0.9668    0.9510       271
         Mild DR     0.5072    0.6250    0.5600        56
     Moderate DR     0.7467    0.3733    0.4978       150
       Severe DR     0.2059    0.4828    0.2887        29
Proliferative DR     0.3103    0.4091    0.3529        44

        accuracy                         0.7000       550
       macro avg     0.5412    0.5714    0.5301       550
    weighted avg     0.7520    0.7000    0.7048       550



In [24]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix")
print("----------------")
print(cm)


Confusion Matrix
----------------
[[262   8   1   0   0]
 [  6  35   9   1   5]
 [  9  23  56  39  23]
 [  0   0   3  14  12]
 [  3   3   6  14  18]]
